In [17]:
from datasets import load_from_disk
from gensim.corpora import Dictionary
from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast
import yaml
import numpy as np
import torch.nn.functional as F
import torch
import torch.nn as nn
import copy
from torch.nn.utils.rnn import pad_sequence

with open('./../../configs/transformer.yaml', 'r') as f:
  config = yaml.safe_load(f)

dataset = load_from_disk("../../data/processed/ast_BPE")
code_dictionary = Dictionary.load('./../../data/processed/ast_BPE/code_dictionary.pt')

tokenizer = PreTrainedTokenizerFast(tokenizer_file="../../data/processed/ast_BPE/bpe_tokenizer.json",
                                    pad_token="[PAD]",
                                    bos_token="[BOS]",
                                    eos_token="[EOS]",
                                    unk_token="[UNK]")

code_dictionary.id2token = {
    v: k for k, v in code_dictionary.token2id.items()
}

In [18]:
torch.cuda.empty_cache() # clears GPU memory
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, input_dim=None, proj_values=True):
        super().__init__()
        self.linear_out = nn.Linear(n_heads * d_model, d_model)
        self.attn_heads = nn.ModuleList([Attention(d_model,
                                                   input_dim=input_dim,
                                                   proj_values=proj_values)
                                         for _ in range(n_heads)])

    def init_keys(self, key):
        for attn in self.attn_heads:
            attn.init_keys(key)

    @property
    def alphas(self):
        # Shape: n_heads, N, 1, L (source)
        return torch.stack([attn.alphas for attn in self.attn_heads], dim=0)

    def output_function(self, contexts):
        # N, 1, n_heads * D
        concatenated = torch.cat(contexts, axis=-1)
        out = self.linear_out(concatenated) # N, 1, D
        return out

    def forward(self, query, mask=None):
        contexts = [attn(query, mask=mask) for attn in self.attn_heads]
        out = self.output_function(contexts)
        return out

class Attention(nn.Module):
    def __init__(self, hidden_dim, input_dim=None, proj_values=False):
        super().__init__()
        self.d_k = hidden_dim
        self.input_dim = hidden_dim if input_dim is None else input_dim
        self.proj_values = proj_values
        self.linear_query = nn.Linear(self.input_dim, hidden_dim)
        self.linear_key = nn.Linear(self.input_dim, hidden_dim)
        self.linear_value = nn.Linear(self.input_dim, hidden_dim)
        self.alphas = None

    def init_keys(self, keys):
        self.keys = keys
        self.proj_keys = self.linear_key(self.keys)
        self.values = self.linear_value(self.keys) \
                      if self.proj_values else self.keys

    def score_function(self, query):
        proj_query = self.linear_query(query)
        # scaled dot product
        # N, 1, H x N, H, L -> N, 1, L
        dot_products = torch.bmm(proj_query, self.proj_keys.permute(0, 2, 1))
        scores =  dot_products / np.sqrt(self.d_k)
        return scores

    def forward(self, query, mask=None):
        # Query is batch-first N, 1, H
        scores = self.score_function(query) # N, 1, L
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        alphas = F.softmax(scores, dim=-1) # N, 1, L
        self.alphas = alphas.detach()

        # N, 1, L x N, L, H -> N, 1, H
        context = torch.bmm(alphas, self.values)
        return context

In [20]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, n_heads, d_model, dropout=0.1):
        super(MultiHeadedAttention, self).__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = int(d_model / n_heads)
        self.linear_query = nn.Linear(d_model, d_model)
        self.linear_key = nn.Linear(d_model, d_model)
        self.linear_value = nn.Linear(d_model, d_model)
        self.linear_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p=dropout)
        self.alphas = None

    def make_chunks(self, x):
        batch_size, seq_len = x.size(0), x.size(1)
        # N, L, D -> N, L, n_heads * d_k
        x = x.view(batch_size, seq_len, self.n_heads, self.d_k)
        # N, n_heads, L, d_k
        x = x.transpose(1, 2)
        return x

    def init_keys(self, key):
        # N, n_heads, L, d_k
        self.proj_key = self.make_chunks(self.linear_key(key))
        self.proj_value = self.make_chunks(self.linear_value(key))

    def score_function(self, query):
        # scaled dot product
        # N, n_heads, L, d_k x # N, n_heads, d_k, L -> N, n_heads, L, L
        proj_query = self.make_chunks(self.linear_query(query))
        dot_products = torch.matmul(proj_query,
                                    self.proj_key.transpose(-2, -1))
        scores =  dot_products / np.sqrt(self.d_k)
        return scores

    def attn(self, query, mask=None):
        # Query is batch-first: N, L, D
        # Score function will generate scores for each head
        scores = self.score_function(query) # N, n_heads, L, L
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        alphas = F.softmax(scores, dim=-1) # N, n_heads, L, L
        alphas = self.dropout(alphas)
        self.alphas = alphas.detach()

        # N, n_heads, L, L x N, n_heads, L, d_k -> N, n_heads, L, d_k
        context = torch.matmul(alphas, self.proj_value)
        return context

    def output_function(self, contexts):
        # N, L, D
        out = self.linear_out(contexts) # N, L, D
        return out

    def forward(self, query, mask=None):
        if mask is not None:
            # N, 1, L, L - every head uses the same mask
            mask = mask.unsqueeze(1)

        # N, n_heads, L, d_k
        context = self.attn(query, mask=mask)
        # N, L, n_heads, d_k
        context = context.transpose(1, 2).contiguous()
        # N, L, n_heads * d_k = N, L, d_model
        context = context.view(query.size(0), -1, self.d_model)
        # N, L, d_model
        out = self.output_function(context)
        return out

In [21]:
class SubLayerWrapper(nn.Module):
    def __init__(self, d_model, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, sublayer, is_self_attn=False, **kwargs):
        norm_x = self.norm(x)
        if is_self_attn:
            sublayer.init_keys(norm_x)
        out = x + self.drop(sublayer(norm_x, **kwargs))
        return out

In [22]:
class EncoderSelfAttn(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, n_features=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.ff_units = ff_units
        self.n_features = n_features
        self.self_attn_heads = MultiHeadAttention(n_heads, d_model, input_dim=n_features)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_units),
            nn.ReLU(),
            nn.Linear(ff_units, d_model),
        )

    def forward(self, query, mask=None):
        self.self_attn_heads.init_keys(query)
        att = self.self_attn_heads(query, mask)
        out = self.ffn(att)
        return out

class DecoderSelfAttn(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, n_features=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.ff_units = ff_units
        self.n_features = d_model if n_features is None else n_features
        self.self_attn_heads = MultiHeadAttention(n_heads, d_model, input_dim=self.n_features)
        self.cross_attn_heads = MultiHeadAttention(n_heads, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_units),
            nn.ReLU(),
            nn.Linear(ff_units, self.n_features),
        )

    def init_keys(self, states):
        self.cross_attn_heads.init_keys(states)

    def forward(self, query, source_mask=None, target_mask=None):
        self.self_attn_heads.init_keys(query)
        att1 = self.self_attn_heads(query, target_mask)
        att2 = self.cross_attn_heads(att1, source_mask)
        out = self.ffn(att2)
        return out

In [23]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        self.d_model = d_model
        self.register_buffer('pe', self._create_pe(max_len, d_model))

    def _create_pe(self, length, d_model):
        """Funzione helper per generare la matrice dei seni e coseni"""
        pe = torch.zeros(length, d_model)
        position = torch.arange(0, length).float().unsqueeze(1)
        angular_speed = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * angular_speed)
        pe[:, 1::2] = torch.cos(position * angular_speed)
        return pe.unsqueeze(0) # Dimensioni: (1, L, D)

    def forward(self, x):
        seq_len = x.size(1)
        
        # CONTROLLO DINAMICO: se x è più lungo di pe, espando pe
        if seq_len > self.pe.size(1):
            # Creiamo una nuova PE lunga quanto serve (o anche un po' di più per sicurezza)
            new_pe = self._create_pe(seq_len, self.d_model).to(x.device)
            self.register_buffer('pe', new_pe)

        scaled_x = x * np.sqrt(self.d_model)
        # Ora lo slicing [:seq_len] non andrà mai più in errore
        encoded = scaled_x + self.pe[:, :seq_len, :]
        return encoded

In [24]:
class EncoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.ff_units = ff_units
        self.self_attn_heads = MultiHeadedAttention(n_heads, d_model,
                                                    dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_units, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, query, mask=None):
        # Sublayer #0
        # Norm
        norm_query = self.norm1(query)
        # Multi-headed Attention
        self.self_attn_heads.init_keys(norm_query)
        states = self.self_attn_heads(norm_query, mask)
        # Add
        att = query + self.drop1(states)

        # Sublayer #1
        # Norm
        norm_att = self.norm2(att)
        # Feed Forward
        out = self.ffn(norm_att)
        # Add
        out = att + self.drop2(out)
        return out
    
class EncoderTransf(nn.Module):
    def __init__(self, encoder_layer, n_layers=1):
        super().__init__()
        self.d_model = encoder_layer.d_model
        self.pe = PositionalEncoding(self.d_model)
        self.norm = nn.LayerNorm(self.d_model)
        self.layers = nn.ModuleList([copy.deepcopy(encoder_layer)
                                     for _ in range(n_layers)])

    def forward(self, query, mask=None):
        # Positional Encoding
        x = self.pe(query)
        for layer in self.layers:
            x = layer(x, mask)
        # Norm
        return self.norm(x)

In [25]:
class DecoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, ff_units, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.ff_units = ff_units
        self.self_attn_heads = MultiHeadedAttention(n_heads, d_model,
                                                    dropout=dropout)
        self.cross_attn_heads = MultiHeadedAttention(n_heads, d_model,
                                                     dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_units, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)
        self.drop3 = nn.Dropout(dropout)

    def init_keys(self, states):
        self.cross_attn_heads.init_keys(states)

    def forward(self, query, source_mask=None, target_mask=None):
        # Sublayer #0
        # Norm
        norm_query = self.norm1(query)
        # Masked Multi-head Attention
        self.self_attn_heads.init_keys(norm_query)
        states = self.self_attn_heads(norm_query, target_mask)
        # Add
        att1 = query + self.drop1(states)

        # Sublayer #1
        # Norm
        norm_att1 = self.norm2(att1)
        # Multi-head Attention
        encoder_states = self.cross_attn_heads(norm_att1, source_mask)
        # Add
        att2 = att1 + self.drop2(encoder_states)

        # Sublayer #2
        # Norm
        norm_att2 = self.norm3(att2)
        # Feed Forward
        out = self.ffn(norm_att2)
        # Add
        out = att2 + self.drop3(out)
        return out
    
class DecoderTransf(nn.Module):
    def __init__(self, decoder_layer, n_layers=1):
        super(DecoderTransf, self).__init__()
        self.d_model = decoder_layer.d_model
        self.pe = PositionalEncoding(self.d_model)
        self.norm = nn.LayerNorm(self.d_model)
        self.layers = nn.ModuleList([copy.deepcopy(decoder_layer)
                                     for _ in range(n_layers)])

    def init_keys(self, states):
        for layer in self.layers:
            layer.init_keys(states)

    def forward(self, query, source_mask=None, target_mask=None):
        # Positional Encoding
        x = self.pe(query)
        for layer in self.layers:
            x = layer(x, source_mask, target_mask)
        # Norm
        return self.norm(x)

In [26]:
class EncoderDecoderSelfAttn(nn.Module):
    def __init__(self, encoder, decoder, max_pred_len):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.max_pred_len = max_pred_len

    @staticmethod
    def subsequent_mask(size):
        attn_shape = (1, size, size)
        subsequence_mask = (1 - torch.triu(torch.ones(attn_shape), diagonal=1))
        return subsequence_mask

    def encode(self, source_seq, source_mask):
        encoder_states = self.encoder(source_seq, source_mask)
        self.decoder.init_keys(encoder_states)

    def decode(self, shifted_target_seq, source_mask=None, target_mask=None):
        outputs = self.decoder(shifted_target_seq,
                               source_mask=source_mask,
                               target_mask=target_mask)
        return outputs

    def predict(self, source_seq, source_mask):
        batch_size = source_seq.shape[0]
        # initialize with bos
        inputs = torch.full((batch_size, 1), tokenizer.bos_token_id, 
                            device=source_seq.device).long()
        for i in range(self.max_pred_len):
          current_sz = inputs.size(1)
          mask = self.subsequent_mask(current_sz).to(inputs.device).bool()
          out = self.decode(inputs, source_mask, mask)
          
        
          # 3. Prendiamo l'ultima parola predetta
          # Usiamo dim=-1 per l'argmax sul vocabolario
          predicted_id = out[:, -1:, :].argmax(dim=-1) 
          
          # 4. Concateniamo (dim=1 è la lunghezza della sequenza)
          inputs = torch.cat([inputs, predicted_id], dim=1)
          if (predicted_id == tokenizer.eos_token_id).all():
              break
          
        outputs = inputs[:, 1:]
        return outputs

    def forward(self, input_seq, target_seq=None, source_mask=None):
        """
        input_seq: indici del codice sorgente
        target_seq: indici del commento (opzionale, usato solo in training)
        """
        # 1. Encoding del codice
        self.encode(input_seq, source_mask)

        # 2. Scelta tra Training (decodifica parallela) o Inference (un token alla volta)
        if target_seq is not None:
            # In training usiamo il commento "shifted" (insegnante forzato)
            # Se target_seq è [BOS, A, B, C, EOS], diamo in pasto al decoder [BOS, A, B, C]
            # La maschera si occupa di non far vedere il futuro
            sz = target_seq.shape[1]
        
            # 2. Creiamo la maschera "al volo"
            # È FONDAMENTALE passargli il device del modello
            device = target_seq.device 
            mask = self.subsequent_mask(sz).to(device).bool()
            outputs = self.decode(target_seq, source_mask, mask)
        else:
            print("not training")
            # In test/predizione generiamo autoregressivamente
            outputs = self.predict(input_seq, source_mask)

        return outputs

In [27]:
class EncoderDecoderTransf(EncoderDecoderSelfAttn):
    def __init__(self, encoder, decoder, src_vocab_size, tgt_vocab_size, max_pred_len=100):
        super(EncoderDecoderTransf, self).__init__(encoder, decoder, max_pred_len)
        self.src_embed = nn.Embedding(src_vocab_size, encoder.d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, decoder.d_model)
        self.linear = nn.Linear(decoder.d_model, tgt_vocab_size)

    def encode(self, source_seq, source_mask=None):
        # Projection
        source_embedded = self.src_embed(source_seq)
        encoder_states = self.encoder(source_embedded, source_mask)
        self.decoder.init_keys(encoder_states)

    def decode(self, shifted_target_seq, source_mask=None, target_mask=None):
        # Projection
        target_embedded = self.tgt_embed(shifted_target_seq)
        outputs = self.decoder(target_embedded,
                               source_mask=source_mask,
                               target_mask=target_mask)
        # Linear
        outputs = self.linear(outputs)
        return outputs

In [28]:
class CollateFn:
  def __init__(self, src_pad_id, tgt_pad_id):
    self.src_pad_id = src_pad_id
    self.tgt_pad_id = tgt_pad_id

  def __call__(self, batch):
    input_ids = [torch.tensor(x['input_ids'], dtype=torch.long) for x in batch]
    labels = []
    for x in batch:
      seq = [tokenizer.bos_token_id]+x['labels']+[tokenizer.eos_token_id]
      labels.append(torch.tensor(seq, dtype=torch.long))

    input_ids = pad_sequence(
      input_ids,
      batch_first=True,
      padding_value=self.src_pad_id
    )

    labels = pad_sequence(
      labels,
      batch_first=True,
      padding_value=self.tgt_pad_id
    )

    return input_ids, labels
  
collate = CollateFn(
  src_pad_id=code_dictionary.token2id['[PAD]'],
  tgt_pad_id=tokenizer.pad_token_id
)

In [29]:
import os

def save_checkpoint(epoch, model, optimizer, val_loss):
    os.makedirs(config['checkpoint_dir'], exist_ok=True)
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    torch.save({'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                }, checkpoint_path)
  
def load_checkpoint(model, optimizer):
    checkpoint_path = f'{config["checkpoint_dir"]}/best_model.pt'
    if(not os.path.exists(checkpoint_path)):
        print("No checkpoint found, starting from scratch")
        return 0
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch']
    print(f"Loaded checkpoint from epoch {start_epoch}")

    for state in optimizer.state.values():
      for k, v in state.items():
          if isinstance(v, torch.Tensor):
              state[k] = v.to(device)
    return start_epoch

In [30]:
from torch.utils.data import DataLoader
generator = torch.Generator()
generator.manual_seed(42)
torch.manual_seed(42)

train_dataset = dataset['train'].select(range(config['train_data_length']))
valid_dataset = dataset['valid'].select(range(config['valid_data_length']))

batch_size = config['batch_size']

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    collate_fn=collate,
    generator=generator
)

In [31]:
torch.manual_seed(42)
# Layers
enclayer = EncoderLayer(n_heads=config['n_heads'], d_model=config['d_model'], ff_units=config['ff_units'], dropout=config['dropout'])
declayer = DecoderLayer(n_heads=config['n_heads'], d_model=config['d_model'], ff_units=config['ff_units'], dropout=config['dropout'])
# Encoder and Decoder
enctransf = EncoderTransf(enclayer, n_layers=config['n_layers'])
dectransf = DecoderTransf(declayer, n_layers=config['n_layers'])
# Transformer
model = EncoderDecoderTransf(
  enctransf, 
  dectransf, 
  src_vocab_size=len(code_dictionary.token2id), 
  tgt_vocab_size=tokenizer.vocab_size,
  max_pred_len=100)
loss = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

In [32]:

epochs = config['num_epochs']
start_epoch = load_checkpoint(model, optimizer)+1

#device = 'cpu'
model.to(device)
best_loss = float('inf')
counter = 0

for epoch in range(start_epoch, epochs):
  model.train()
  batch_losses = []

  for input, labels in train_dataloader:
    input, labels = input.to(device), labels.to(device)
    optimizer.zero_grad()

    # Prepariamo gli input per il Teacher Forcing
    dec_input = labels[:, :-1]
    targets = labels[:, 1:]

    # Maschera per il codice (ignora il padding)
    source_mask = (input != code_dictionary.token2id['[PAD]']).unsqueeze(1)

    # Forward: nota che passiamo dec_input e non labels intero
    y_pred = model(input, dec_input, source_mask=source_mask)
    
    # Loss: confrontiamo con i targets (shiftati di 1)
    single_loss = loss(y_pred.permute(0, 2, 1), targets)
    
    single_loss.backward()
    optimizer.step()
    batch_losses.append(single_loss.item())

  
  if(epoch % 5 == 0):
    print(f"Epoch {epoch:3}, Training Loss: {sum(batch_losses)/len(batch_losses):10.8f}")

  model.eval()
  with torch.no_grad():
    val_losses = []

    for input, labels in valid_dataloader:
        input, labels = input.to(device), labels.to(device)
        
        dec_input = labels[:, :-1]
        targets = labels[:, 1:]
        
        src_mask = (input != code_dictionary.token2id['[PAD]']).unsqueeze(1)
        
        y_pred = model(input_seq=input, target_seq=dec_input, source_mask=src_mask)
        v_loss = loss(y_pred.permute(0, 2, 1), targets)
        val_losses.append(v_loss)

  if epoch % 5 == 0:
    print(f"Epoch {epoch:3}, Validation Loss: {sum(val_losses)/len(val_losses):10.8f}")

  if(len(val_losses) > 0 and sum(val_losses)/len(val_losses) < best_loss):
    best_loss = sum(val_losses)/len(val_losses)
    counter = 0
    save_checkpoint(epoch, model, optimizer, best_loss)
    print(f"Saved new best model (epoch {epoch})")
    
  else:
    counter += 1
    if counter >= config['patience']:
      print("Early stopping")
      break
  
  torch.cuda.empty_cache() # clears GPU memory
  del single_loss
  del y_pred
  del input
  del labels

No checkpoint found, starting from scratch
Saved new best model (epoch 1)
Saved new best model (epoch 2)
Epoch   5, Training Loss: 5.19432652
Epoch   5, Validation Loss: 6.23524427
Early stopping


In [33]:
def predict_ids(model, input_seq, max_length=50):
    if len(input_seq) == 0:
        return None
    model.eval()
    device = next(model.parameters()).device

    # Trasformiamo in tensor [1, L]
    input_tensor = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)
    
    # 1. Calcoliamo l'encoding una sola volta (Memory)
    # Rimuovi i riferimenti a hidden/cell
    source_mask = (input_tensor != code_dictionary.token2id['[PAD]']).unsqueeze(1)
    with torch.no_grad():
        encoder_memory = model.encode(input_tensor, source_mask)

    # 2. Inizializziamo la sequenza col token BOS
    # Per il Transformer, 'inputs' crescerà nel loop: [1, 1] -> [1, 2] -> [1, 3]...
    inputs = torch.tensor([[tokenizer.bos_token_id]], dtype=torch.long).to(device)

    pred_ids = []
    for i in range(max_length):
        # 3. Creiamo la maschera per la sequenza generata finora
        curr_sz = inputs.size(1)
        trg_mask = model.subsequent_mask(curr_sz).to(device).bool()

        # 4. Decoder Forward
        # Passiamo tutta la sequenza 'inputs' e la 'encoder_memory'
        with torch.no_grad():
            out = model.decode(inputs, source_mask, trg_mask) # Assicurati che riceva anche encoder_memory se necessario dal tuo metodo
            
            # Prendiamo i logits dell'ULTIMO step temporale
            logits = out[:, -1, :]
            
            # 5. Sampling (Multinomial) come facevi prima
            temperature = 0.7
            probs = torch.softmax(logits / temperature, dim=-1)
            next_token_id = torch.multinomial(probs, 1).item()

        if next_token_id == tokenizer.eos_token_id:
            break
            
        pred_ids.append(next_token_id)
        
        # 6. IMPORTANTE: Concateniamo il nuovo token a quelli precedenti
        next_token_tensor = torch.tensor([[next_token_id]], dtype=torch.long).to(device)
        inputs = torch.cat([inputs, next_token_tensor], dim=1)
    
    return pred_ids
  
def decode_ids(pred_ids):
  return [tokenizer.decode(token_id) for token_id in pred_ids]

test_dataset = dataset['test'].select(range(config['test_data_length']))

test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate
)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

smooth_fn = SmoothingFunction().method4
bleu_scores = []
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
rouge_scores = []

test_dataset = dataset['test'].select(range(config['test_data_length']))

load_checkpoint(model, optimizer)
model.eval()
model.to(device)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=True,
    collate_fn=collate
)


for(input, labels) in test_dataloader:
  for i in range(input.size(0)):
    with torch.no_grad():
      pred_ids = predict_ids(model, input[i].tolist())
    
    if pred_ids == None:
       continue
    pred_tokens  = decode_ids(pred_ids)
    labels_tokens = decode_ids(labels[i].tolist())

    print("Predicted Docstring: ", ' '.join(pred_tokens))
    print("Actual Docstring:    ", ' '.join(labels_tokens))

    bleu_score = sentence_bleu(
        [labels_tokens],
        pred_tokens,
        smoothing_function=smooth_fn
    )
    rouge_score = rouge.score(' '.join(labels_tokens), ' '.join(pred_tokens))['rougeL'].fmeasure

    bleu_scores.append(bleu_score)
    rouge_scores.append(rouge_score)

print(f"Average BLEU score on test set: {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average RougeL score on test set: {sum(rouge_scores)/len(rouge_scores):.4f}")

Loaded checkpoint from epoch 2
Predicted Docstring:  returns a and return the part .
Actual Docstring:     [BOS] str - > list convert xml to url list . from bi li grab . [EOS]
Predicted Docstring:  set the a a given the vhost .
Actual Docstring:     [BOS] downloads daily motion videos by url . [EOS]
Predicted Docstring:  serialize the file .
Actual Docstring:     [BOS] downloads sin a videos by url . [EOS]
Predicted Docstring:  calculate the file from a timestamp
Actual Docstring:     [BOS] format text with color or other effects into ansi escaped string . [EOS]
Predicted Docstring:  transform the numbers of the appropriate exceptions .
Actual Docstring:     [BOS] print a log message to standard error . [EOS]
Predicted Docstring:  also a list and update the ture .
Actual Docstring:     [BOS] print an error log message . [EOS]
Predicted Docstring:  place a an over the dataset .
Actual Docstring:     [BOS] what a ter ri ble failure ! [EOS]
Predicted Docstring:  retrieve all the correct t